# Assignment 2: Household objects + synthetic video frames (Step-Video-T2V)

Topic from A1: books, cups, pens, spoons.
SOTA add-on: short clips made offline with **Step-Video-T2V**, sampled into frames and added to the training set.
All generated frames are **synthetic** (file names start with `syn_`). They contain no people or faces.

Steps:
1. Generate clips on yuewen.cn/videos (see PROMPTS.md), save them as mp4 in `MyDrive/compvision2_videos/<Class>/`.
2. Run this notebook from top to bottom.
3. Copy `app.py` to `MyDrive/compvision2_results/` and run the last cell.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip -q install gradio opencv-python-headless

In [ ]:
import os, random, shutil, time, platform
import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image, ImageEnhance, ImageFilter
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, ConcatDataset, Subset, Dataset
from torchvision import datasets, transforms, models
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, confusion_matrix, average_precision_score

random.seed(42)
np.random.seed(42)
torch.manual_seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"
print("Device:", gpu)

In [ ]:
BASE = os.environ.get("A2_BASE", "/content/drive/MyDrive")
REAL = BASE + "/compvision1_cropped"
VIDEOS = BASE + "/compvision2_videos"
FRAMES = BASE + "/compvision2_frames"
OUT = BASE + "/compvision2_results"
SPLIT = os.environ.get("A2_SPLIT", "/content/split2")
classes = ["Books", "Cups", "Pens", "Spoons"]
EPOCHS = 15
FRAMES_PER_VIDEO = 4
os.makedirs(OUT, exist_ok=True)

## 1. Frames from the generated clips

In [ ]:
def crop60(img):
    w, h = img.size
    nw, nh = int(w * 0.6), int(h * 0.6)
    left, top = (w - nw) // 2, (h - nh) // 2
    return img.crop((left, top, left + nw, top + nh))

def grab(path, n):
    cap = cv2.VideoCapture(path)
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    idx = np.linspace(0, total - 1, n).astype(int)
    out = []
    for i in idx:
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(i))
        ok, fr = cap.read()
        if ok:
            out.append(cv2.cvtColor(fr, cv2.COLOR_BGR2RGB))
    cap.release()
    return out

if os.path.exists(FRAMES):
    shutil.rmtree(FRAMES)

for c in classes:
    os.makedirs(FRAMES + "/" + c)
    vids = sorted(f for f in os.listdir(VIDEOS + "/" + c) if f.lower().endswith(".mp4"))
    for k, v in enumerate(vids):
        frames = grab(VIDEOS + "/" + c + "/" + v, FRAMES_PER_VIDEO)
        for j, fr in enumerate(frames):
            crop60(Image.fromarray(fr)).save(f"{FRAMES}/{c}/syn_{k:03d}_{j:02d}.jpg", quality=95)
    print(c, len(vids), "videos,", len(os.listdir(FRAMES + "/" + c)), "frames")

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for ax, c in zip(axes, classes):
    f = sorted(os.listdir(FRAMES + "/" + c))[0]
    ax.imshow(Image.open(FRAMES + "/" + c + "/" + f))
    ax.set_title("SYNTHETIC: " + c)
    ax.axis("off")
plt.tight_layout()
plt.savefig(OUT + "/synthetic_samples.png", dpi=150)
plt.show()

## 2. Dataset split (real images only in val and test)

In [ ]:
if os.path.exists(SPLIT):
    shutil.rmtree(SPLIT)

for s in ["train", "val", "test"]:
    for c in classes:
        os.makedirs(SPLIT + "/" + s + "/" + c)

rows = []
for c in classes:
    imgs = sorted(f for f in os.listdir(REAL + "/" + c) if not f.startswith("."))
    random.Random(42).shuffle(imgs)
    n = len(imgs)
    n_train = int(0.70 * n)
    n_val = int(0.15 * n)
    for i, f in enumerate(imgs):
        if i < n_train:
            s = "train"
        elif i < n_train + n_val:
            s = "val"
        else:
            s = "test"
        shutil.copy(REAL + "/" + c + "/" + f, SPLIT + "/" + s + "/" + c + "/" + f)
        rows.append([c, f, s])

pd.DataFrame(rows, columns=["class", "file", "split"]).to_csv(OUT + "/split.csv", index=False)
for s in ["train", "val", "test"]:
    print(s, {c: len(os.listdir(SPLIT + "/" + s + "/" + c)) for c in classes})

In [ ]:
IMG = 224
norm = transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])

train_tf = transforms.Compose([
    transforms.Resize((IMG, IMG)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    norm
])

eval_tf = transforms.Compose([
    transforms.Resize((IMG, IMG)),
    transforms.ToTensor(),
    norm
])

train_real = datasets.ImageFolder(SPLIT + "/train", transform=train_tf)
val_ds = datasets.ImageFolder(SPLIT + "/val", transform=eval_tf)
test_ds = datasets.ImageFolder(SPLIT + "/test", transform=eval_tf)
syn_ds = datasets.ImageFolder(FRAMES, transform=train_tf)

assert train_real.classes == syn_ds.classes == classes
print("Real train:", len(train_real), "| Val:", len(val_ds), "| Test:", len(test_ds), "| Synthetic frames:", len(syn_ds))

## 3. Model, training, metrics

In [ ]:
def make_model():
    m = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
    m.fc = nn.Linear(m.fc.in_features, 4)
    return m.to(device)

def loader(ds, shuffle):
    return DataLoader(ds, batch_size=32, shuffle=shuffle, num_workers=2)

def fit(model, tl, vl, epochs=EPOCHS, lr=1e-4):
    crit = nn.CrossEntropyLoss()
    opt = optim.Adam(model.parameters(), lr=lr)
    sched = optim.lr_scheduler.StepLR(opt, step_size=5, gamma=0.5)
    hist = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}
    best = -1
    best_w = None
    for ep in range(epochs):
        model.train()
        tl_sum, tc, tn = 0, 0, 0
        for x, y in tl:
            x, y = x.to(device), y.to(device)
            opt.zero_grad()
            out = model(x)
            loss = crit(out, y)
            loss.backward()
            opt.step()
            tl_sum += loss.item() * x.size(0)
            tc += (out.argmax(1) == y).sum().item()
            tn += x.size(0)
        model.eval()
        vl_sum, vc, vn = 0, 0, 0
        with torch.no_grad():
            for x, y in vl:
                x, y = x.to(device), y.to(device)
                out = model(x)
                vl_sum += crit(out, y).item() * x.size(0)
                vc += (out.argmax(1) == y).sum().item()
                vn += x.size(0)
        sched.step()
        hist["train_loss"].append(tl_sum / tn)
        hist["val_loss"].append(vl_sum / vn)
        hist["train_acc"].append(tc / tn)
        hist["val_acc"].append(vc / vn)
        if vc / vn > best:
            best = vc / vn
            best_w = {k: v.clone() for k, v in model.state_dict().items()}
        print(f"Epoch {ep + 1:02d}/{epochs} | Train {tl_sum / tn:.4f} {tc / tn:.4f} | Val {vl_sum / vn:.4f} {vc / vn:.4f}")
    model.load_state_dict(best_w)
    return hist

def predict(model, ds):
    model.eval()
    ys, ps = [], []
    with torch.no_grad():
        for x, y in DataLoader(ds, batch_size=32):
            ps.append(model(x.to(device)).softmax(1).cpu().numpy())
            ys.extend(y.tolist())
    return np.array(ys), np.concatenate(ps)

def score(y, p):
    pred = p.argmax(1)
    return {
        "acc": accuracy_score(y, pred),
        "prec": precision_score(y, pred, average="macro", zero_division=0),
        "rec": recall_score(y, pred, average="macro", zero_division=0),
        "f1": f1_score(y, pred, average="macro"),
        "mAP": average_precision_score(np.eye(4)[y], p, average="macro"),
    }

def latency(model, runs=100):
    model.eval()
    x = torch.randn(1, 3, IMG, IMG).to(device)
    with torch.no_grad():
        for _ in range(10):
            model(x)
        if device.type == "cuda":
            torch.cuda.synchronize()
        t = time.time()
        for _ in range(runs):
            model(x)
        if device.type == "cuda":
            torch.cuda.synchronize()
    ms = (time.time() - t) / runs * 1000
    return ms, 1000 / ms

## 4. Stress test: accuracy by condition

The test images are the same real photos, but we make them blurry, dark, rotated or noisy.
This is how the brief asks to check robustness (motion, blur, viewpoint, lighting).

In [ ]:
CONDS = ["clean", "blur", "dark", "rotate", "noise"]

def shift(img, kind, seed):
    if kind == "blur":
        return img.filter(ImageFilter.GaussianBlur(4))
    if kind == "dark":
        return ImageEnhance.Brightness(img).enhance(0.35)
    if kind == "rotate":
        return img.rotate(30)
    if kind == "noise":
        a = np.array(img).astype(np.float32)
        a = a + np.random.RandomState(seed).normal(0, 25, a.shape)
        return Image.fromarray(a.clip(0, 255).astype(np.uint8))
    return img

class Sh(Dataset):
    def __init__(self, base, kind):
        self.base = base
        self.kind = kind

    def __len__(self):
        return len(self.base.samples)

    def __getitem__(self, i):
        path, y = self.base.samples[i]
        img = Image.open(path).convert("RGB")
        return eval_tf(shift(img, self.kind, i)), y

## 5. Experiment and ablation

Same real train / val / test split every time. Only the share of synthetic frames in the training set changes:
0% is the A1 baseline (ResNet18 on real images), 100% uses all extracted frames.

In [ ]:
FRACS = [0.0, 0.5, 1.0]
res, stress, hists, trained = [], [], {}, {}

for fr in FRACS:
    print("\n===== synthetic fraction", fr, "=====")
    torch.manual_seed(42)
    n = int(len(syn_ds) * fr)
    idx = random.Random(42).sample(range(len(syn_ds)), n)
    train_ds = ConcatDataset([train_real, Subset(syn_ds, idx)]) if n > 0 else train_real
    model = make_model()
    hists[fr] = fit(model, loader(train_ds, True), loader(val_ds, False))
    trained[fr] = model
    y, p = predict(model, test_ds)
    s = score(y, p)
    ms, fps = latency(model)
    s.update({"synthetic_fraction": fr, "synthetic_frames": n, "train_size": len(train_ds),
              "synthetic_share": round(n / len(train_ds), 3), "latency_ms": round(ms, 2), "FPS": round(fps, 1)})
    res.append(s)
    for k in CONDS:
        yk, pk = predict(model, Sh(test_ds, k))
        stress.append({"synthetic_fraction": fr, "condition": k, "acc": accuracy_score(yk, pk.argmax(1))})

res_df = pd.DataFrame(res)
stress_df = pd.DataFrame(stress).pivot(index="synthetic_fraction", columns="condition", values="acc")[CONDS]
res_df.to_csv(OUT + "/results.csv", index=False)
stress_df.to_csv(OUT + "/stress.csv")
print("\nEnvironment:", gpu, "| torch", torch.__version__, "|", platform.platform())
res_df.round(4)

In [ ]:
stress_df.round(4)

In [ ]:
base = trained[0.0]
sota = trained[1.0]
torch.save(base.state_dict(), OUT + "/resnet18_baseline.pth")
torch.save(sota.state_dict(), OUT + "/resnet18_sota.pth")
for k, h in hists.items():
    pd.DataFrame(h).to_csv(OUT + f"/history_{k}.csv", index=False)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for k, name in [(0.0, "Baseline (real only)"), (1.0, "Real + synthetic")]:
    axes[0].plot(hists[k]["train_loss"], label=name + " train")
    axes[0].plot(hists[k]["val_loss"], "--", label=name + " val")
    axes[1].plot(hists[k]["train_acc"], label=name + " train")
    axes[1].plot(hists[k]["val_acc"], "--", label=name + " val")
axes[0].set_title("Loss")
axes[1].set_title("Accuracy")
for a in axes:
    a.set_xlabel("Epoch")
    a.legend()
plt.tight_layout()
plt.savefig(OUT + "/training_curves.png", dpi=150)
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, k, name in zip(axes, [0.0, 1.0], ["Baseline (real only)", "Real + synthetic"]):
    y, p = predict(trained[k], test_ds)
    sns.heatmap(confusion_matrix(y, p.argmax(1)), annot=True, fmt="d", cmap="Blues",
                xticklabels=classes, yticklabels=classes, ax=ax)
    ax.set_title(name)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")
plt.tight_layout()
plt.savefig(OUT + "/confusion_matrices.png", dpi=150)
plt.show()

stress_df.T.plot(kind="bar", figsize=(8, 4))
plt.ylabel("Accuracy")
plt.title("Accuracy by condition (legend = synthetic fraction)")
plt.tight_layout()
plt.savefig(OUT + "/stress.png", dpi=150)
plt.show()

## 6. Explainability: Grad-CAM

In [ ]:
def cam(model, x):
    store = {}

    def fh(m, i, o):
        store["a"] = o
        o.register_hook(lambda g: store.__setitem__("g", g))

    h = model.layer4.register_forward_hook(fh)
    model.eval()
    out = model(x)
    c = out.argmax(1).item()
    model.zero_grad()
    out[0, c].backward()
    h.remove()
    w = store["g"].mean(dim=(2, 3), keepdim=True)
    m = torch.relu((w * store["a"]).sum(1)).squeeze(0).detach().cpu().numpy()
    return m / (m.max() + 1e-8), out.softmax(1)[0].detach().cpu().numpy()

def overlay(img, m):
    h = cv2.resize(m, (img.shape[1], img.shape[0]))
    h = cv2.applyColorMap((h * 255).astype(np.uint8), cv2.COLORMAP_JET)
    h = cv2.cvtColor(h, cv2.COLOR_BGR2RGB) / 255.0
    return (0.5 * img + 0.5 * h).clip(0, 1)

mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
ids = random.Random(1).sample(range(len(test_ds)), 6)

fig, axes = plt.subplots(2, 6, figsize=(18, 6))
for col, i in enumerate(ids):
    x, y = test_ds[i]
    m, p = cam(sota, x.unsqueeze(0).to(device))
    img = (x * std + mean).permute(1, 2, 0).numpy().clip(0, 1)
    axes[0, col].imshow(img)
    axes[0, col].set_title("True: " + classes[y])
    axes[1, col].imshow(overlay(img, m))
    axes[1, col].set_title(f"Pred: {classes[p.argmax()]} {p.max():.2f}")
    axes[0, col].axis("off")
    axes[1, col].axis("off")
plt.tight_layout()
plt.savefig(OUT + "/gradcam.png", dpi=150)
plt.show()

## 7. Run the app

Copy `app.py` into `MyDrive/compvision2_results/` first. The link opens in a new tab.

In [ ]:
import sys
sys.path.insert(0, OUT)
os.environ["MODEL_PATH"] = OUT + "/resnet18_sota.pth"
import app
app.demo.launch(share=True)